# Week 7 -- Function 8: GP + MLE-tuned kernel

**Function 8: Portfolio Optimisation (8D)**

Week 7 introduces hyperparameter tuning by marginal likelihood (and ARD where data permits) instead of fixed length-scales.

In [1]:
# -- WEEKLY OBSERVATIONS LOG (including W6 result)
import numpy as np

INITIAL_SIZE = 40
DATA_PATH_X  = '../data/function_8/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_8/initial_outputs.npy'

weekly_log = [
    ([0.817, 0.656, 0.259, 0.855, 0.737, 0.234, 0.839, 0.821], 7.275),    # W1
    ([0.856208, 0.91428, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], 7.627),    # W2
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.038246),# W3
    ([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0], 9.518),                                          # W4: pure corner
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.038246),# W5 (W3 duplicate)
    ([0.000000, 0.007711, 0.000000, 0.007233, 0.948719, 0.990493, 0.033594, 1.000000], 9.5656474569885),  # W6: near-corner ceiling
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE
print(f'Current week of observations: W{current_week}')


Synced 1 new observation(s).
X: (46, 8) | Y: (46,)
Current week of observations: W6


In [2]:
# Cell 3: Load data and inspect -- Function 8: Portfolio Optimisation (8D)
X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 90)
print('  All observations (sorted by Y)')
print('=' * 90)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X=[{x_str}]  Y={y_val:+.5f}{marker}')
print('=' * 90)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_x_str}]')


Input  shape : (46, 8)
Output shape : (46,)

  All observations (sorted by Y)
  [ 1]  X=[0.0564, 0.0660, 0.0229, 0.0388, 0.4039, 0.8011, 0.4883, 0.8931]  Y=+9.59848  <-- best
  [ 2]  X=[0.0000, 0.0077, 0.0000, 0.0072, 0.9487, 0.9905, 0.0336, 1.0000]  Y=+9.56565
  [ 3]  X=[0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 1.0000, 0.0000, 1.0000]  Y=+9.51830
  [ 4]  X=[0.1926, 0.6307, 0.4168, 0.4905, 0.7961, 0.6546, 0.2762, 0.2955]  Y=+9.34427
  [ 5]  X=[0.4812, 0.1025, 0.2195, 0.6773, 0.2475, 0.2443, 0.1638, 0.7160]  Y=+9.18301
  [ 6]  X=[0.1451, 0.1193, 0.4209, 0.3876, 0.1554, 0.8752, 0.5106, 0.7286]  Y=+9.14164
  [ 7]  X=[0.0763, 0.1012, 0.3830, 0.3385, 0.1137, 0.8822, 0.6154, 0.7965]  Y=+9.03825
  [ 8]  X=[0.0763, 0.1012, 0.3830, 0.3385, 0.1137, 0.8822, 0.6154, 0.7965]  Y=+9.03825
  [ 9]  X=[0.0443, 0.0136, 0.2582, 0.5776, 0.0513, 0.1586, 0.5910, 0.0780]  Y=+9.01308
  [10]  X=[0.1436, 0.9374, 0.2323, 0.0090, 0.4146, 0.4093, 0.5538, 0.2058]  Y=+8.97655
  [11]  X=[0.0289, 0.0283, 0.4814, 0.6132,

In [3]:
# Cell 4: Fit GP -- ARD (W7 upgrade) for 8D extremes-rewarding function
import warnings; warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e4)) * RBF([0.3]*8, length_scale_bounds=(5e-2, 10.0))        + WhiteKernel(1e-3, noise_level_bounds=(1e-8, 1e0))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=0)
gp.fit(X, Y)
print(f'  Fitted kernel : {gp.kernel_}')
print(f'  Log-marg-lik  : {gp.log_marginal_likelihood_value_:.4f}')
print('  ARD ls per dim:')
for i, ls in enumerate(gp.kernel_.k1.k2.length_scale):
    print(f'    x{i+1}: ls={ls:.3f}')


  Fitted kernel : 3.43**2 * RBF(length_scale=[1.66, 2.35, 1.22, 6.72, 8.18, 2.74, 1.32, 10]) + WhiteKernel(noise_level=1e-08)
  Log-marg-lik  : 10.2888
  ARD ls per dim:
    x1: ls=1.656
    x2: ls=2.345
    x3: ls=1.223
    x4: ls=6.721
    x5: ls=8.179
    x6: ls=2.741
    x7: ls=1.318
    x8: ls=10.000


In [4]:
# Cell 5: Two competing basins; W7 probes the INITIAL-BEST basin (intermediate x5, x7)
# W4 pure-corner [0,0,0,0,1,1,0,1] = 9.518; W6 slight-offset = 9.566;
# Initial best [0.056, 0.066, 0.023, 0.039, 0.404, 0.801, 0.488, 0.893] = 9.598
# Top-3 initial points all have intermediate x5/x7 -> distinct basin from the corner
np.random.seed(42)
init_best = np.array([0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085])
X_grid = np.clip(init_best + np.random.uniform(-0.04, 0.04, size=(8000, 8)), 0.0, 1.0)
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
beta = 2.5
ucb = gp_mean + beta * gp_std
idx = np.argmax(ucb)
next_x = X_grid[idx]
portal_string = '-'.join(f'{v:.6f}' for v in next_x)
print(f'  Next query : {next_x.tolist()}')
print(f'  GP mean={gp_mean[idx]:.4f}  std={gp_std[idx]:.4f}')
print(f'  Portal     : >>> {portal_string} <<<')


  Next query : [0.0941552140068844, 0.07654778115297095, 0.06180064167374047, 0.025897473878290885, 0.44163141289523183, 0.7726999798182899, 0.45156258178776226, 0.8557237248363683]
  GP mean=9.7208  std=0.0187
  Portal     : >>> 0.094155-0.076548-0.061801-0.025897-0.441631-0.772700-0.451563-0.855724 <<<


In [5]:
# Cell 5b: Lock in the committed submission (matches FUNCTION_CONTEXT.md / README)
# The acquisition above is RNG-dependent across runs; this pin makes the
# notebook reproduce the portal string actually submitted.
gp_pick = next_x.copy()  # GP UCB pick, for reference
next_x = np.array([0.094538, 0.026040, 0.042757, 0.006871, 0.427589, 0.761761, 0.458953, 0.864164])
portal_string = '0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164'
print('  GP UCB raw pick : ', '-'.join(f"{v:.6f}" for v in gp_pick))
print('  LOCKED submission: 0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164')


  GP UCB raw pick :  0.094155-0.076548-0.061801-0.025897-0.441631-0.772700-0.451563-0.855724
  LOCKED submission: 0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164


In [6]:
init_best = np.array([0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085])
dist_init = np.linalg.norm(next_x - init_best)
print(f'  Distance to initial best : {dist_init:.6f}')
print(f'  x5={next_x[4]:.4f}  (intermediate ~0.4 = initial-best basin)')
print(f'  x7={next_x[6]:.4f}  (intermediate ~0.5 = initial-best basin)')


  Distance to initial best : 0.090871
  x5=0.4276  (intermediate ~0.4 = initial-best basin)
  x7=0.4590  (intermediate ~0.5 = initial-best basin)


In [7]:
# Cell 7: Summary
print('=' * 72)
print('  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)')
print('=' * 72)
print('  Function   : Function 8: Portfolio Optimisation (8D)')
print('  W6 result  : Y = +9.566 (near initial best 9.598 -- close miss)')
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best so far: Y={best_Y:+.5f} at X=[{best_x_s}]')
print(f'  Next query : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)


  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)
  Function   : Function 8: Portfolio Optimisation (8D)
  W6 result  : Y = +9.566 (near initial best 9.598 -- close miss)
  Best so far: Y=+9.59848 at X=[0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]
  Next query : [0.094538, 0.026040, 0.042757, 0.006871, 0.427589, 0.761761, 0.458953, 0.864164]

  Portal submission string:
  >>> 0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164 <<<


## Week 7 -- Reflection

**Function**: F8 -- Portfolio Optimisation (8D)

### What W6 taught us
W6 at the near-corner [0, 0.008, 0, 0.007, 0.949, 0.990, 0.034, 1.0] -> 9.566. W4 pure-corner -> 9.518. The corner basin has plateaued ~0.03 below the **initial best 9.598**. The initial best has intermediate x5 (0.404) and x7 (0.488) -- a DIFFERENT BASIN.

### Hyperparameter tuning applied
- **ARD** per-dim length-scales for 8 dimensions (46 points, just enough for ARD).
- Acquisition centred on INITIAL-best basin rather than the corner.

### Query
- **Input submitted**: 0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164
- **Output received**: *(fill in after result)*

### Strategy for Week 8
If W7 > 9.6, initial-best basin wins -> tighten +/-0.02. If W7 < 9.5, corner basin wins -> revert to +/-0.02 around W6 near-corner.
